In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display
from groq import Groq

In [2]:
load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

Groq API Key exists and begins gsk_


In [3]:
openai = Groq()

groq_url = "https://api.groq.com/openai/v1"

groq = Groq(api_key=groq_api_key, base_url=groq_url)

In [4]:
GROQ_MODEL_1="llama-3.1-8b-instant"
GROQ_MODEL_2="llama-3.3-70b-versatile"

In [5]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'mingw32'},
 'package_managers': ['winget'],
 'cpu': {'brand': 'Intel(R) Core(TM) i5-8350U CPU @ 1.70GHz',
  'cores_logical': 8,
  'cores_physical': 4,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.exe (MinGW.org GCC-6.3.0-1) 6.3.0',
   'g++': 'g++.exe (MinGW.org GCC-6.3.0-1) 6.3.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [6]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=GROQ_MODEL_2, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

Based on the provided system information, you already have a C++ compiler installed (`g++.exe` from MinGW). You do not need to install any additional compilers.

To compile and execute your `main.cpp` file using Python, you can use the following commands:

```python
import subprocess

# Compile command
compile_command = ["g++", "-O3", "-o", "main.exe", "main.cpp"]

# Compile and store the result
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)

# Run command
run_command = ["./main.exe"]

# Run the compiled executable and store the result
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)

return run_result.stdout
```

Here's a brief explanation of the compile command:

- `g++`: The C++ compiler you're using (in this case, the one that comes with MinGW).
- `-O3`: This option enables the compiler to perform optimizations to improve the performance of the compiled code.
- `-o main.exe`: This specifies the output file name for the compiled executable.
- `main.cpp`: The input file to be compiled.

The `run_command` simply executes the compiled executable (`main.exe`). The `./` is necessary because the executable is in the current working directory, and Windows requires the directory separator to be specified explicitly. 

Note that this code assumes that the `main.cpp` file is in the same directory as the Python script. If it's located elsewhere, you'll need to provide the full path to the file in the `compile_command` and `run_command`. 

Also, ensure that the directory containing the `g++.exe` compiler is included in your system's PATH environment variable. If it's not, you can either add it to the PATH or provide the full path to the compiler in the `compile_command`. 

It's worth noting that the compiled executable will only work on Windows platforms, as it's being compiled using MinGW, which targets the Windows platform.

In [41]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["main.exe"]

In [8]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [9]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [10]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [11]:
def port(client, model, python):
    reasoning_effort = "high" if 'groq' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [35]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200000000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [36]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [14]:
run_python(pi)

Result: 3.141592656089
Execution Time: 122.285861 seconds


In [37]:
port(openai, GROQ_MODEL_2, pi)

In [49]:
import subprocess

def run_cpp():
    compile_command = ["g++", "-O3", "-o", "main.exe", "main.cpp"]

    compile_result = subprocess.run(
        compile_command,
        text=True,
        capture_output=True
    )

    print("Return code:", compile_result.returncode)

    if compile_result.stdout:
        print("STDOUT:")
        print(compile_result.stdout)

    if compile_result.stderr:
        print("STDERR:")
        print(compile_result.stderr)

    if compile_result.returncode != 0:
        return

    run_result = subprocess.run(
        [r".\main.exe"],
        text=True,
        capture_output=True,
        shell=True  
        
    )

    print(run_result.stdout)


In [50]:
run_cpp()

Return code: 0
Result: 3.141592656090
Execution Time: 0.603989 seconds



In [53]:
122.285861/0.603989

202.46372202142754

In [54]:
#202 times fasteer